_Developer Note:_ if you may make a PR in the future, be sure to copy this
notebook, and use the `gitignore` prefix `temp` to avoid future conflicts.

This is one notebook in a multi-part series on clusterless decoding in Spyglass

- To set up your Spyglass environment and database, see
  [the Setup notebook](./00_Setup.ipynb)
- For additional info on DataJoint syntax, including table definitions and
  inserts, see
  [the Insert Data notebook](./01_Insert_Data.ipynb)
- Prior to running, please familiarize yourself with the [spike sorting
  pipeline](./02_Spike_Sorting.ipynb) and generate input position data with
  either the [Trodes](./20_Position_Trodes.ipynb) or DLC notebooks
  ([1](./21_Position_DLC_1.ipynb), [2](./22_Position_DLC_2.ipynb),
  [3](./23_Position_DLC_3.ipynb)).

The goal of this notebook is to populate the `UnitWaveformFeatures` table, which depends `SpikeSortingOutput`. This table contains the features of the waveforms of each unit.

While clusterless decoding avoids actual spike sorting, we need to pass through these tables to maintain (relative) pipeline simplicity. Pass-through tables keep spike sorting and clusterless waveform extraction as similar as possible, by using shared steps. Here, "spike sorting" involves simple thresholding (sorter: clusterless_thresholder).


In [ ]:
from pathlib import Path
import datajoint as dj

dj.config.load(
    Path("../dj_local_conf.json").absolute()
)  # load config for database connection info

First, if you haven't inserted the `minirec20230622.nwb` file into the database, you should do so now. This is the file that we will use for the decoding tutorials.

> **Binder hub:** the upstream tutorial uses `mediumnwb20230802.nwb` (32 GB), which is too large for this hub. Here we use the 42-second `minirec` recording instead, so every step runs, but the extracted features and decoding results are only illustrative.


In [ ]:
from spyglass.utils.nwb_helper_fn import get_nwb_copy_filename
import spyglass.data_import as sgi
import spyglass.position as sgp

# Insert the nwb file
nwb_file_name = "minirec20230622.nwb"  # Binder hub: was mediumnwb20230802.nwb (32 GB)
nwb_copy_file_name = get_nwb_copy_filename(nwb_file_name)
sgi.insert_sessions(nwb_file_name)

# Position
sgp.v1.TrodesPosParams.insert_default()

interval_list_name = "pos 0 valid times"

trodes_s_key = {
    "nwb_file_name": nwb_copy_file_name,
    "interval_list_name": interval_list_name,
    "trodes_pos_params_name": "single_led",  # Binder hub: minirec params from 20_Position_Trodes
}
sgp.v1.TrodesPosSelection.insert1(
    trodes_s_key,
    skip_duplicates=True,
)
sgp.v1.TrodesPosV1.populate(trodes_s_key)

These next steps are the same as in the [Spike Sorting notebook](./10_Spike_SortingV1.ipynb), but we'll repeat them here for clarity. These are pre-processing steps that are shared between spike sorting and clusterless decoding.

We first set the `SortGroup` to define which contacts are sorted together.

We then setup for spike sorting by bandpass filtering and whitening the data via the `SpikeSortingRecording` table.


In [ ]:
import spyglass.spikesorting.v1 as sgs

# Binder hub: 10_Spike_SortingV1 already made these groups for minirec. Calling
# set_group_by_shank again would offer to delete them and everything sorted from them.
if not (sgs.SortGroup & {"nwb_file_name": nwb_copy_file_name}):
    sgs.SortGroup.set_group_by_shank(nwb_file_name=nwb_copy_file_name)

sort_group_ids = (sgs.SortGroup & {"nwb_file_name": nwb_copy_file_name}).fetch(
    "sort_group_id"
)
# Binder hub: with 1 CPU, all 32 minirec tetrodes take ~25 min to process.
# Use the first 8 (~6 min); set N_SORT_GROUPS = None to use all of them.
N_SORT_GROUPS = 8
sort_group_ids = sort_group_ids[:N_SORT_GROUPS]

group_keys = []
for sort_group_id in sort_group_ids:
    key = {
        "nwb_file_name": nwb_copy_file_name,
        "sort_group_id": sort_group_id,
        "interval_list_name": interval_list_name,
        "preproc_param_name": "default",
        "team_name": "My Team",  # Binder hub: team created in 02_Insert_Data
    }
    group_keys.append(key)
    sgs.SpikeSortingRecordingSelection.insert_selection(key)

sgs.SpikeSortingRecording.populate(group_keys)

Next we do artifact detection. Here we skip it by setting the `artifact_param_name` to `None`, but in practice you should detect artifacts as it will affect the decoding.


In [ ]:
recording_ids = (
    sgs.SpikeSortingRecordingSelection & {"nwb_file_name": nwb_copy_file_name}
).fetch("recording_id")

group_keys = []
for recording_id in recording_ids:
    key = {
        "recording_id": recording_id,
        "artifact_param_name": "none",
    }
    group_keys.append(key)
    sgs.ArtifactDetectionSelection.insert_selection(key)

sgs.ArtifactDetection.populate(group_keys)

Now we run the "spike sorting", which in our case is simply thresholding the signal to find spikes. We use the `SpikeSorting` table to store the results. Note that `sorter_param_name` defines the parameters for thresholding the signal.


In [ ]:
(
    sgs.SpikeSorterParameters()
    & {"sorter": "clusterless_thresholder", "sorter_param_name": "default_clusterless"}
).fetch1()

# Binder hub: minirec's spikes are small, so the default 100 µV threshold
# finds none. Use the low threshold from Spyglass's own minirec tests.
sgs.SpikeSorterParameters.insert1(
    {
        "sorter": "clusterless_thresholder",
        "sorter_param_name": "low_thresh",
        "sorter_params": {
            **(
                sgs.SpikeSorterParameters
                & {"sorter": "clusterless_thresholder", "sorter_param_name": "default_clusterless"}
            ).fetch1("sorter_params"),
            "detect_threshold": 10.0,  # default 100
            "local_radius_um": 1000,  # default 100
        },
    },
    skip_duplicates=True,
)

In [ ]:
group_keys = []
for recording_id in recording_ids:
    key = {
        "recording_id": recording_id,
        "sorter": "clusterless_thresholder",
        "sorter_param_name": "low_thresh",  # Binder hub: see above
        "nwb_file_name": nwb_copy_file_name,
        "interval_list_name": str(
            (
                sgs.ArtifactDetectionSelection
                & {"recording_id": recording_id, "artifact_param_name": "none"}
            ).fetch1("artifact_id")
        ),
    }
    group_keys.append(key)
    sgs.SpikeSortingSelection.insert_selection(key)
sort_keys = (sgs.SpikeSortingSelection & group_keys).fetch("KEY")
sgs.SpikeSorting.populate(sort_keys)

For clusterless decoding we do not need any manual curation, but for the sake of the pipeline, we need to store the output of the thresholding in the `CurationV1` table and insert this into the `SpikeSortingOutput` table.


In [ ]:
sgs.SpikeSorting().populate(
    sgs.SpikeSortingSelection
    & {
        "nwb_file_name": nwb_copy_file_name,
        "sorter": "clusterless_thresholder",
        "sorter_param_name": "low_thresh",
    }
)

In [ ]:
from spyglass.spikesorting.spikesorting_merge import SpikeSortingOutput

sorting_ids = (
    sgs.SpikeSortingSelection
    & {
        "nwb_file_name": nwb_copy_file_name,
        "sorter": "clusterless_thresholder",
        "sorter_param_name": "low_thresh",
    }
).fetch("sorting_id")

for sorting_id in sorting_ids:
    try:
        sgs.CurationV1.insert_curation(sorting_id=sorting_id)
    except KeyError:
        pass

SpikeSortingOutput.insert(
    sgs.CurationV1().fetch("KEY"),
    part_name="CurationV1",
    skip_duplicates=True,
)

Finally, we extract the waveform features of each SortGroup. This is done by the `UnitWaveformFeatures` table.

To set this up, we use the `WaveformFeaturesParams` to define the time around the spike that we want to use for feature extraction, and which features to extract. Here is an example of the parameters used for extraction the amplitude of the negative peak of the waveform:

```python

waveform_extraction_params = {
    "ms_before": 0.5,
    "ms_after": 0.5,
    "max_spikes_per_unit": None,
    "n_jobs": 5,
    "total_memory": "5G",
}
waveform_feature_params = {
    "amplitude": {
        "peak_sign": "neg",
        "estimate_peak_time": False,
    }
}
```

We see that we want 0.5 ms of time before and after the peak of the negative spike. We also see that we want to extract the amplitude of the negative peak, and that we do not want to estimate the peak time (since we know it is at 0 ms).

You can define other features to extract such as spatial location of the spike:

```python
waveform_extraction_params = {
    "ms_before": 0.5,
    "ms_after": 0.5,
    "max_spikes_per_unit": None,
    "n_jobs": 5,
    "total_memory": "5G",
}
waveform_feature_params = {
    "amplitude": {
        "peak_sign": "neg",
        "estimate_peak_time": False,
    },
    "spike location": {}
}

```

_Note_: Members of the Frank Lab can use "ampl_10_jobs_v2" instead of "amplitude"
for significant speed improvements.


In [ ]:
from spyglass.decoding.v1.waveform_features import WaveformFeaturesParams

waveform_extraction_params = {
    "ms_before": 0.5,
    "ms_after": 0.5,
    "max_spikes_per_unit": None,
    "n_jobs": 1,  # Binder hub: 1 CPU / 2 GB per server (upstream: 5 jobs, "5G")
    "total_memory": "500M",
}
waveform_feature_params = {
    "amplitude": {
        "peak_sign": "neg",
        "estimate_peak_time": False,
    }
}

WaveformFeaturesParams.insert1(
    {
        "features_param_name": "amplitude",
        "params": {
            "waveform_extraction_params": waveform_extraction_params,
            "waveform_feature_params": waveform_feature_params,
        },
    },
    skip_duplicates=True,
)

WaveformFeaturesParams()

Now that we've inserted the waveform features parameters, we need to define which parameters to use for each SortGroup. This is done by the `UnitWaveformFeaturesSelection` table. We need to link the primary key `merge_id` from the `SpikeSortingOutput` table to a features parameter set.


In [ ]:
from spyglass.decoding.v1.waveform_features import UnitWaveformFeaturesSelection

UnitWaveformFeaturesSelection()

First we find the units we need.  We can use the method `SpikeSortingOutput.get_restricted_merge_ids()` to perform the needed joins to find them:


In [ ]:
nwb_copy_file_name = "minirec20230622_.nwb"
from spyglass.spikesorting.spikesorting_merge import SpikeSortingOutput

In [ ]:
key = {
    "nwb_file_name": nwb_copy_file_name,
    "sorter": "clusterless_thresholder",
    "sorter_param_name": "low_thresh",
}
merge_ids = SpikeSortingOutput().get_restricted_merge_ids(
    {
        "nwb_file_name": nwb_copy_file_name,
        "sorter": "clusterless_thresholder",
        "sorter_param_name": "low_thresh",
    },
    sources=["v1"],
)

Then we link them with the features parameters:


In [ ]:
selection_keys = [
    {
        "spikesorting_merge_id": merge_id,
        "features_param_name": "amplitude",
    }
    for merge_id in merge_ids
]
UnitWaveformFeaturesSelection.insert(selection_keys, skip_duplicates=True)

UnitWaveformFeaturesSelection & selection_keys

Finally, we extract the waveform features, by populating the `UnitWaveformFeatures` table:


In [ ]:
from spyglass.decoding.v1.waveform_features import UnitWaveformFeatures

UnitWaveformFeatures.populate(selection_keys)

In [ ]:
UnitWaveformFeatures & selection_keys

Now that we've extracted the data, we can inspect the results. Let's fetch the data:


In [ ]:
spike_times, spike_waveform_features = (
    UnitWaveformFeatures & selection_keys
).fetch_data()

Let's look at the features shape. This is a list corresponding to tetrodes, with each element being a numpy array of shape (n_spikes, n_features). The features in this case are the amplitude of each tetrode wire at the negative peak of the waveform.


In [ ]:
for features in spike_waveform_features:
    print(features.shape)

We can plot the amplitudes to see if there is anything that looks neural and to look for outliers:


In [ ]:
import matplotlib.pyplot as plt

tetrode_ind = 1
plt.scatter(
    spike_waveform_features[tetrode_ind][:, 0],
    spike_waveform_features[tetrode_ind][:, 1],
    s=1,
)